# Лабораторна робота №2 — Частина 2

## Individual Household Electric Power Consumption Dataset

У цій частині роботи виконується очищення та аналіз датасету споживання електроенергії домогосподарством.

Ноутбук автоматично завантажує датасет у папку `data/`, тому вручну завантажувати файл не потрібно.


## Завдання 1

Завантажити та відкрити датасет **Individual Household Electric Power Consumption Dataset**.

Файл з даними не додається до GitHub, оскільки він великий. Під час виконання ноутбука він автоматично завантажується локально.


In [1]:
from pathlib import Path
import timeit
import urllib.request
import zipfile

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, StandardScaler

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

ZIP_PATH = DATA_DIR / "household_power_consumption.zip"
DATA_PATH = DATA_DIR / "household_power_consumption.txt"

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"

def download_dataset() -> Path:
    """Завантажує та розпаковує датасет, якщо він ще не завантажений."""
    if DATA_PATH.exists():
        print(f"Датасет уже існує: {DATA_PATH}")
        return DATA_PATH

    print("Завантаження датасету...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

    print("Розпакування датасету...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(DATA_DIR)

    if not DATA_PATH.exists():
        raise FileNotFoundError("Після розпакування файл household_power_consumption.txt не знайдено.")

    print(f"Датасет готовий: {DATA_PATH}")
    return DATA_PATH

download_dataset()


Завантаження датасету...
Розпакування датасету...
Датасет готовий: data/household_power_consumption.txt


PosixPath('data/household_power_consumption.txt')

## Завдання 2

Зчитати датасет у pandas DataFrame та виконати data cleaning.

У цьому датасеті пропущені значення позначені символом `?`, тому під час зчитування вони перетворюються на `NaN`.


In [2]:
def load_power_consumption_dataset(path: Path) -> pd.DataFrame:
    """Завантажує та очищує датасет споживання електроенергії."""
    df = pd.read_csv(
        path,
        sep=";",
        na_values="?",
        low_memory=False,
    )

    df["datetime"] = pd.to_datetime(
        df["Date"] + " " + df["Time"],
        format="%d/%m/%Y %H:%M:%S",
        errors="coerce",
    )

    numeric_columns = [
        "Global_active_power",
        "Global_reactive_power",
        "Voltage",
        "Global_intensity",
        "Sub_metering_1",
        "Sub_metering_2",
        "Sub_metering_3",
    ]

    for column in numeric_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    df = df.dropna(subset=["datetime"])
    df[numeric_columns] = df[numeric_columns].fillna(df[numeric_columns].median())

    df["date"] = df["datetime"].dt.date
    df["time"] = df["datetime"].dt.time
    df["hour"] = df["datetime"].dt.hour
    df["month"] = df["datetime"].dt.month
    df["day_of_week"] = df["datetime"].dt.day_name()

    return df

power_df = load_power_consumption_dataset(DATA_PATH)
power_df.head()


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime,date,time,hour,month,day_of_week
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00,2006-12-16,17:24:00,17,12,Saturday
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00,2006-12-16,17:25:00,17,12,Saturday
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00,2006-12-16,17:26:00,17,12,Saturday
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00,2006-12-16,17:27:00,17,12,Saturday
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00,2006-12-16,17:28:00,17,12,Saturday


In [3]:
power_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 15 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   Date                   object        
 1   Time                   object        
 2   Global_active_power    float64       
 3   Global_reactive_power  float64       
 4   Voltage                float64       
 5   Global_intensity       float64       
 6   Sub_metering_1         float64       
 7   Sub_metering_2         float64       
 8   Sub_metering_3         float64       
 9   datetime               datetime64[ns]
 10  date                   object        
 11  time                   object        
 12  hour                   int32         
 13  month                  int32         
 14  day_of_week            object        
dtypes: datetime64[ns](1), float64(7), int32(2), object(5)
memory usage: 221.7+ MB


## Завдання 3

Окремими функціями сформувати вибірки згідно з умовою лабораторної роботи.


In [4]:
def select_active_power_over_5kw(df: pd.DataFrame) -> pd.DataFrame:
    """Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт."""
    return df[df["Global_active_power"] > 5].copy()

def select_intensity_19_20_and_submetering_condition(df: pd.DataFrame) -> pd.DataFrame:
    """
    Обрати записи, у яких сила струму лежить в межах 19–20 А.
    Для них вибрати ті, де Sub_metering_2 і Sub_metering_3 разом більші за Sub_metering_1.
    """
    filtered = df[df["Global_intensity"].between(19, 20, inclusive="both")].copy()
    condition = (filtered["Sub_metering_2"] + filtered["Sub_metering_3"]) > filtered["Sub_metering_1"]
    return filtered[condition].copy()

def random_sample_500000_and_mean_submetering(df: pd.DataFrame, random_state: int = 42) -> pd.Series:
    """Обрати випадкові 500000 записів без повторів і обчислити середні для 3 груп споживання."""
    sample_size = min(500_000, len(df))
    sample = df.sample(n=sample_size, replace=False, random_state=random_state)
    return sample[["Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]].mean()

def select_evening_high_consumption_complex(df: pd.DataFrame) -> pd.DataFrame:
    """
    Обрати записи після 18:00 з потужністю понад 6 кВт.
    Серед них залишити записи, де Sub_metering_2 є найбільшою групою.
    Потім обрати кожен третій результат із першої половини та кожен четвертий із другої половини.
    """
    filtered = df[(df["hour"] >= 18) & (df["Global_active_power"] > 6)].copy()
    filtered = filtered[
        (filtered["Sub_metering_2"] > filtered["Sub_metering_1"])
        & (filtered["Sub_metering_2"] > filtered["Sub_metering_3"])
    ].copy()

    midpoint = len(filtered) // 2
    first_half = filtered.iloc[:midpoint].iloc[::3]
    second_half = filtered.iloc[midpoint:].iloc[::4]

    return pd.concat([first_half, second_half]).reset_index(drop=True)

def normalize_and_standardize(df: pd.DataFrame, columns: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Повертає нормалізований і стандартизований DataFrame для вказаних колонок."""
    minmax_scaler = MinMaxScaler()
    standard_scaler = StandardScaler()

    normalized = pd.DataFrame(
        minmax_scaler.fit_transform(df[columns]),
        columns=[f"{col}_normalized" for col in columns],
        index=df.index,
    )

    standardized = pd.DataFrame(
        standard_scaler.fit_transform(df[columns]),
        columns=[f"{col}_standardized" for col in columns],
        index=df.index,
    )

    return normalized, standardized

def calculate_correlations(df: pd.DataFrame, column_1: str, column_2: str) -> pd.Series:
    """Підрахувати коефіцієнти Пірсона та Спірмена для двох числових атрибутів."""
    return pd.Series(
        {
            "pearson": df[column_1].corr(df[column_2], method="pearson"),
            "spearman": df[column_1].corr(df[column_2], method="spearman"),
        }
    )

def one_hot_encode_column(df: pd.DataFrame, column: str) -> pd.DataFrame:
    """Виконати One Hot Encoding категоріального атрибута."""
    return pd.get_dummies(df, columns=[column], prefix=column)


## Завдання 4

Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.


In [5]:
result_1 = select_active_power_over_5kw(power_df)
result_1.head(), result_1.shape


(          Date      Time  Global_active_power  Global_reactive_power  Voltage  \
 1   16/12/2006  17:25:00                5.360                  0.436   233.63   
 2   16/12/2006  17:26:00                5.374                  0.498   233.29   
 3   16/12/2006  17:27:00                5.388                  0.502   233.74   
 11  16/12/2006  17:35:00                5.412                  0.470   232.78   
 12  16/12/2006  17:36:00                5.224                  0.478   232.99   
 
     Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  \
 1               23.0             0.0             1.0            16.0   
 2               23.0             0.0             2.0            17.0   
 3               23.0             0.0             1.0            17.0   
 11              23.2             0.0             1.0            17.0   
 12              22.4             0.0             1.0            16.0   
 
               datetime        date      time  hour  month day_of_

## Завдання 5

Обрати всі записи, у яких сила струму лежить в межах 19–20 А, і відібрати ті, де одна група споживання переважає іншу згідно з умовою.


In [6]:
result_2 = select_intensity_19_20_and_submetering_condition(power_df)
result_2.head(), result_2.shape


(          Date      Time  Global_active_power  Global_reactive_power  Voltage  \
 10  16/12/2006  17:34:00                4.448                  0.498   232.86   
 24  16/12/2006  17:48:00                4.474                  0.000   234.96   
 33  16/12/2006  17:57:00                4.512                  0.000   233.62   
 45  16/12/2006  18:09:00                4.464                  0.136   234.66   
 52  16/12/2006  18:16:00                4.524                  0.076   234.20   
 
     Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  \
 10              19.6             0.0             1.0            17.0   
 24              19.4             0.0             0.0            17.0   
 33              19.2             0.0             0.0            17.0   
 45              19.0             0.0            37.0            16.0   
 52              19.6             0.0             9.0            17.0   
 
               datetime        date      time  hour  month day_of_

## Завдання 6

Обрати випадковим чином 500000 записів без повторів та обчислити середні величини усіх 3-х груп споживання електроенергії.


In [7]:
result_3 = random_sample_500000_and_mean_submetering(power_df)
result_3


,0
Sub_metering_1,1.109002
Sub_metering_2,1.283718
Sub_metering_3,6.397150


## Завдання 7

Обрати записи після 18:00 з потужністю понад 6 кВт, залишити записи з найбільшим значенням другої групи споживання, а потім вибрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.


In [8]:
result_4 = select_evening_high_consumption_complex(power_df)
result_4.head(), result_4.shape


(         Date      Time  Global_active_power  Global_reactive_power  Voltage  \
 0  16/12/2006  18:05:00                6.052                  0.192   232.93   
 1  16/12/2006  18:08:00                6.308                  0.116   232.25   
 2  28/12/2006  20:58:00                6.386                  0.374   236.63   
 3  28/12/2006  21:02:00                8.088                  0.262   235.50   
 4  28/12/2006  21:05:00                7.230                  0.152   235.22   
 
    Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  \
 0              26.2             0.0            37.0            17.0   
 1              27.0             0.0            36.0            17.0   
 2              27.0             1.0            36.0            17.0   
 3              34.4             1.0            72.0            17.0   
 4              30.6             1.0            73.0            17.0   
 
              datetime        date      time  hour  month day_of_week  
 0 200

## Завдання 8

Пронормувати та стандартизувати вибраний датасет.


In [9]:
numeric_columns_for_scaling = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",
]

normalized_df, standardized_df = normalize_and_standardize(power_df, numeric_columns_for_scaling)
normalized_df.head()


,Global_active_power_normalized,Global_reactive_power_normalized,Voltage_normalized,Global_intensity_normalized,Sub_metering_1_normalized,Sub_metering_2_normalized,Sub_metering_3_normalized
0,0.374796,0.300719,0.376090,0.377593,0.0,0.0125,0.548387
1,0.478363,0.313669,0.336995,0.473029,0.0,0.0125,0.516129
2,0.479631,0.358273,0.326010,0.473029,0.0,0.0250,0.548387
3,0.480898,0.361151,0.340549,0.473029,0.0,0.0125,0.548387
4,0.325005,0.379856,0.403231,0.323651,0.0,0.0125,0.548387


In [10]:
standardized_df.head()


,Global_active_power_standardized,Global_reactive_power_standardized,Voltage_standardized,Global_intensity_standardized,Sub_metering_1_standardized,Sub_metering_2_standardized,Sub_metering_3_standardized
0,2.975591,2.629139,-1.864146,3.120054,-0.181154,-0.048773,1.262163
1,4.062977,2.789788,-2.239958,4.160250,-0.181154,-0.048773,1.143202
2,4.076284,3.343136,-2.345558,4.160250,-0.181154,0.124020,1.262163
3,4.089591,3.378836,-2.205793,4.160250,-0.181154,-0.048773,1.262163
4,2.452810,3.610885,-1.603252,2.532116,-0.181154,-0.048773,1.262163


## Завдання 9

Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.


In [11]:
calculate_correlations(power_df, "Global_active_power", "Global_intensity")


,0
pearson,0.998891
spearman,0.995508


## Завдання 10

Провести One Hot Encoding категоріального атрибута.

Категоріальним атрибутом у цій роботі буде `day_of_week`, який ми отримали з дати.


In [12]:
encoded_sample = one_hot_encode_column(power_df.head(20), "day_of_week")
encoded_sample.head()


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,datetime,date,time,hour,month,day_of_week_Saturday
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00,2006-12-16,17:24:00,17,12,True
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00,2006-12-16,17:25:00,17,12,True
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00,2006-12-16,17:26:00,17,12,True
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00,2006-12-16,17:27:00,17,12,True
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00,2006-12-16,17:28:00,17,12,True


## Завдання 11

Проаналізувати часові витрати на виконання процедур за допомогою модуля `timeit`.


In [13]:
def profile_function(statement: str, globals_dict: dict, number: int = 3) -> float:
    """Повертає середній час виконання statement."""
    total_time = timeit.timeit(statement, globals=globals_dict, number=number)
    return total_time / number

profile_results = pd.DataFrame(
    [
        {
            "operation": "Global_active_power > 5",
            "avg_time_seconds": profile_function(
                "select_active_power_over_5kw(power_df)",
                globals(),
            ),
        },
        {
            "operation": "Global_intensity 19-20 + condition",
            "avg_time_seconds": profile_function(
                "select_intensity_19_20_and_submetering_condition(power_df)",
                globals(),
            ),
        },
        {
            "operation": "Random sample 500000 + mean",
            "avg_time_seconds": profile_function(
                "random_sample_500000_and_mean_submetering(power_df)",
                globals(),
            ),
        },
        {
            "operation": "Evening high consumption complex selection",
            "avg_time_seconds": profile_function(
                "select_evening_high_consumption_complex(power_df)",
                globals(),
            ),
        },
    ]
)

profile_results


,operation,avg_time_seconds
0,Global_active_power > 5,0.014931
1,Global_intensity 19-20 + condition,0.018818
2,Random sample 500000 + mean,0.291344
3,Evening high consumption complex selection,0.016700
